In [1]:
import tensorflow as tf
import cv2
import numpy as np
import os
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Flatten, Dense, Dropout, BatchNormalization, LeakyReLU

In [12]:
# Bengali character mapping
bengali_char_map = {
    0: "অ", 1: "আ", 2: "ই", 3: "ঈ", 4: "উ", 5: "ঊ", 6: "ঋ", 7: "এ", 8: "ঐ", 9: "ও", 10: "ঔ",
    11: "ক", 12: "খ", 13: "গ", 14: "ঘ", 15: "ঙ", 16: "চ", 17: "ছ", 18: "জ", 19: "ঝ", 20: "ঞ",
    21: "ট", 22: "ঠ", 23: "ড", 24: "ঢ", 25: "ণ", 26: "ত", 27: "থ", 28: "দ", 29: "ধ", 30: "ন",
    31: "প", 32: "ফ", 33: "ব", 34: "ভ", 35: "ম", 36: "য", 37: "র", 38: "ল", 39: "শ", 40: "ষ",
    41: "স", 42: "হ", 43: "ড়", 44: "ঢ়", 45: "য়", 46: "ৎ", 47: "ং", 48: "ঃ", 49: "ঁ"
}

In [13]:

# Image preprocessing
def load_and_preprocess_image(image_path):
    image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if image is None:
        print(f"Warning: Unable to read image at {image_path}")
        return None
    image = cv2.resize(image, (32, 32))  # Resize
    image = image / 255.0  # Normalize
    image = np.expand_dims(image, axis=-1)  # Add channel
    return image

In [14]:

# Load dataset
def load_dataset(dataset_path):
    images, labels = [], []
    class_names = sorted(os.listdir(dataset_path))
    valid_extensions = ('.bmp', '.png', '.jpg', '.jpeg')
    
    for label, class_name in enumerate(class_names):
        class_path = os.path.join(dataset_path, class_name)
        for image_name in os.listdir(class_path):
            if image_name.lower().endswith(valid_extensions):
                image_path = os.path.join(class_path, image_name)
                image = load_and_preprocess_image(image_path)
                if image is not None:
                    images.append(image)
                    labels.append(label)
    
    return np.array(images, dtype=np.float32), np.array(labels, dtype=np.int32), class_names


In [16]:

# Dataset paths
train_dataset_path = 'CMATERdb 3.1.2/BasicFinalDatabase/Train'
test_dataset_path = 'CMATERdb 3.1.2/BasicFinalDatabase/Test'



In [17]:
# Load images and labels
train_images, train_labels, train_class_names = load_dataset(train_dataset_path)
test_images, test_labels, test_class_names = load_dataset(test_dataset_path)

In [18]:

# ANN Model
def build_ann_model(input_shape, num_classes):
    model = Sequential([
        Flatten(input_shape=input_shape),
        
        Dense(512),
        BatchNormalization(),
        LeakyReLU(alpha=0.1),
        Dropout(0.4),

        Dense(256),
        BatchNormalization(),
        LeakyReLU(alpha=0.1),
        Dropout(0.4),

        Dense(128),
        BatchNormalization(),
        LeakyReLU(alpha=0.1),
        Dropout(0.3),

        Dense(num_classes, activation='softmax')
    ])
    return model


In [19]:
# Build and compile model
input_shape = (32, 32, 1)
num_classes = len(train_class_names)

model = build_ann_model(input_shape, num_classes)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

d:\image_to_text\.venv\lib\site-packages\keras\src\layers\reshaping\flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
d:\image_to_text\.venv\lib\site-packages\keras\src\layers\activations\leaky_relu.py:41: UserWarning: Argument `alpha` is deprecated. Use `negative_slope` instead.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten (Flatten)               │ (None, 1024)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 512)            │       524,800 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 512)            │         2,048 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu (LeakyReLU)         │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_1 (LeakyReLU)       │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_2 (LeakyReLU)       │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 50)             │         6,450 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 699,058 (2.67 MB)

 Trainable params: 697,266 (2.66 MB)

 Non-trainable params: 1,792 (7.00 KB)

In [20]:
# Train model
model.fit(train_images, train_labels, epochs=30, batch_size=32, validation_split=0.2)


Epoch 1/30
300/300 ━━━━━━━━━━━━━━━━━━━━ 6s 10ms/step - accuracy: 0.1438 - loss: 3.3940 - val_accuracy: 0.0000e+00 - val_loss: 7.3748
Epoch 2/30
300/300 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - accuracy: 0.4317 - loss: 1.9892 - val_accuracy: 0.0000e+00 - val_loss: 8.9434
Epoch 3/30
300/300 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - accuracy: 0.5138 - loss: 1.6573 - val_accuracy: 0.0000e+00 - val_loss: 10.4106
Epoch 4/30
300/300 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.5667 - loss: 1.4593 - val_accuracy: 0.0000e+00 - val_loss: 10.7534
Epoch 5/30
300/300 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - accuracy: 0.5915 - loss: 1.3558 - val_accuracy: 0.0000e+00 - val_loss: 11.0610
Epoch 6/30
300/300 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - accuracy: 0.6165 - loss: 1.2505 - val_accuracy: 0.0000e+00 - val_loss: 11.0779
Epoch 7/30
300/300 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - accuracy: 0.6467 - loss: 1.1611 - val_accuracy: 0.0000e+00 - val_loss: 12.2576
Epoch 8/30
300/300 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - accuracy: 0.6725

In [28]:
# Evaluate
model.evaluate(test_images, test_labels)

94/94 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7078 - loss: 1.3693


[4.283512592315674, 0.5696666836738586]

In [26]:

# Predict from new image
new_image_path = 'CMATERdb 3.1.2/BasicFinalDatabase/Test/174/bcc000051.bmp'
new_image = load_and_preprocess_image(new_image_path)

if new_image is not None:
    prediction = model.predict(np.expand_dims(new_image, axis=0))
    predicted_label = np.argmax(prediction)

    # Get Bengali character
    predicted_text = bengali_char_map.get(predicted_label, "Unknown")
    
    print(f"Predicted label: {predicted_label}, Predicted text: {predicted_text}")
else:
    print("Unable to read the new image.")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step
Predicted label: 2, Predicted text: ই
